# Qcombo `easyCombo` Complete Guide

`easyCombo` is the core entry function of the **qcombo** package, used to compute commutators between many-body operators.

It is based on the **Generalized Wick's Theorem** and automatically handles contraction, regularization, density matrix diagonalization, and expression simplification.
With just one line of code, you can derive the full analytic expression from operators to final results.

---

## Function Signature

```python
qcombo.easyCombo(left, right, contraction=None, latexOutput=None, amcOutput=None, **kwargs)
```

### Parameter Quick Reference

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `left` | int or list | **(required)** | Body rank or indices of the left operator |
| `right` | int or list | **(required)** | Body rank or indices of the right operator |
| `contraction` | int / list / None | `None` | Specify contraction body rank |
| `latexOutput` | str / None | `None` | Custom `.tex` output filename |
| `amcOutput` | str / None | `None` | Custom `.amc` output filename |
| `wick_mode` | `'MR'` / `'SR'` | `'MR'` | Wick contraction mode |
| `show_process` | bool | `True` | Show progress bar and intermediate steps |
| `parallel` | bool | `False` | Enable multi-process parallel computation |
| `savefile` | bool | `True` | Automatically save output files |

---
## 1. Environment Setup

In [3]:
import qcombo
from IPython.display import display, Latex


# Helper function to display expressions in LaTeX format
def jupyterDisplay(expr, title=None):
    """Display a SymPy expression in LaTeX format in Jupyter"""
    if expr == 0 or expr is None:
        display(Latex(r"$$0$$"))
        return
    latex_expr = qcombo.texExp(expr)
    if title:
        print(title)
    display(Latex(f"$${latex_expr}$$"))

print(f"qcombo version: {qcombo.__version__}")

qcombo version: 0.2.0


---
## 2. Minimal Usage: Specify Operator Body Ranks

You can pass **integers** for `left` and `right` — the program automatically assigns indices.

- `easyCombo(1, 1)` → compute $[\text{1B}, \text{1B}]$ commutator
- `easyCombo(2, 2)` → compute $[\text{2B}, \text{2B}]$ commutator

Results are stored in `result.expr_dict`, with `.tex` and `.amc` files auto-saved.

In [4]:
# Compute [1B, 1B] commutator, output all possible contraction body ranks
result = qcombo.easyCombo(1, 1, parallel=False, show_process=False, savefile=False)
# View specific expressions
for key, expr in result.expr_dict.items():
    if expr != 0:
        jupyterDisplay(expr, f"{key}:")

0B_lambda1B:


<IPython.core.display.Latex object>

1B_lambda1B:


<IPython.core.display.Latex object>

---
## 3. `left` / `right` Parameter: Two Input Styles

### (a) Integer form (recommended)

Indices are automatically assigned from `['a','b','c',...]`, suitable for quick derivations.

### (b) String list form

Manually specify upper and lower indices for each operator, suitable when you need specific index naming.

Format: `[ [upper_indices], [lower_indices] ]`

In [5]:
# Method (a): integer
print("=" * 50)
print("(a) Integer form: easyCombo(1, 2)")
print("    1-body operator x 2-body operator, auto-assigned indices")
result_int = qcombo.easyCombo(1, 2, contraction=0,
                              parallel=False, show_process=False, savefile=False)
for key, expr in result_int.expr_dict.items():
    if expr != 0:
        jupyterDisplay(expr, f"  {key}:")

# Method (b): manual indices
print("\n" + "=" * 50)
print("(b) Manual indices: upper=['alpha'], lower=['beta']")
left  = [['alpha'], ['beta']]         # 1-body: upper α, lower β
right = [['i','j'], ['k','l']]         # 2-body: upper i,j, lower k,l
result_manual = qcombo.easyCombo(left, right, contraction=0,
                                 parallel=False, show_process=False, savefile=False)
for key, expr in result_manual.expr_dict.items():
    if expr != 0:
        jupyterDisplay(expr, f"  {key}:")

print("Note: after easyCombo's internal index reassignment, \n the original alpha and beta indices become dummy indices and will not appear in the final expression")

(a) Integer form: easyCombo(1, 2)
    1-body operator x 2-body operator, auto-assigned indices
  0B_lambda2B:


<IPython.core.display.Latex object>


(b) Manual indices: upper=['alpha'], lower=['beta']
  0B_lambda2B:


<IPython.core.display.Latex object>

Note: after easyCombo's internal index reassignment, 
 the original alpha and beta indices become dummy indices and will not appear in the final expression


---
## 4. `contraction` Parameter: Control Output Body Rank

The commutator $[M\text{B}, N\text{B}]$ can contract into $0, 1, \dots, M+N-1$ body terms.

| `contraction` value | Behavior |
|---------------------|----------|
| `None` (default) | Output all possible body ranks |
| `int` (e.g. `0`) | Output only the specified body rank |
| `list` (e.g. `[0,1]`) | Output multiple specified body ranks |

**Valid range**: $0 \le \text{contraction} < M+N$

In [6]:
# Using [2B, 2B] as an example, demonstrate three usages of the contraction parameter

# (1) Default: output all
r_all = qcombo.easyCombo(2, 2, parallel=False, show_process=False, savefile=False)
print("contraction=None (all):", list(r_all.expr_dict.keys()))

# (2) Only output 0-body
r_0b = qcombo.easyCombo(2, 2, contraction=0,
                        parallel=False, show_process=False, savefile=False)
print("contraction=0 (0B only):", list(r_0b.expr_dict.keys()))

# (3) Output 0-body and 1-body
r_01b = qcombo.easyCombo(2, 2, contraction=[0, 1],
                         parallel=False, show_process=False, savefile=False)
print("contraction=[0,1] (0B+1B):", list(r_01b.expr_dict.keys()))

contraction=None (all): ['0B_lambda1B', '0B_lambda2B', '0B_lambda3B', '1B_lambda1B', '1B_lambda2B', '2B_lambda1B', '3B_lambda1B']
contraction=0 (0B only): ['0B_lambda1B', '0B_lambda2B', '0B_lambda3B']
contraction=[0,1] (0B+1B): ['0B_lambda1B', '0B_lambda2B', '0B_lambda3B', '1B_lambda1B', '1B_lambda2B']


---
## 5. `wick_mode` Parameter: SR vs MR Contraction Mode

`wick_mode` controls the reference state type when applying Wick's theorem:

| Mode | Meaning | λ density matrix | Contraction rule |
|------|---------|------------------|------------------|
| `'MR'` (default) | Multi-Reference | Allows multi-body λ | Relaxed |
| `'SR'` | Single-Reference | Only 1-body λ allowed | $\|m-n\| \le k < m+n$|

**MR vs SR comparison example**:

In [7]:
# MR mode
r_mr = qcombo.easyCombo(2, 2, contraction=0,
                        wick_mode='MR', parallel=False,
                        show_process=False, savefile=False)
print("MR mode [2,2]_0 λ classification:")
for key in r_mr.expr_dict:
    print(f"  {key}")

# SR mode
r_sr = qcombo.easyCombo(2, 2, contraction=0,
                        wick_mode='SR', parallel=False,
                        show_process=False, savefile=False)
print("\nSR mode [2,2]_0 λ classification:")
for key in r_sr.expr_dict:
    print(f"  {key}")

print("\nNote: In SR mode, only λ_1B exists — no λ_2B or higher body!")

MR mode [2,2]_0 λ classification:
  0B_lambda1B
  0B_lambda2B
  0B_lambda3B

SR mode [2,2]_0 λ classification:
  0B_lambda1B

Note: In SR mode, only λ_1B exists — no λ_2B or higher body!


---
## 6. `parallel` Parameter: Parallel Computing

| Value | Behavior | Use case |
|-------|----------|----------|
| `True` | Multi-process parallel (ProcessPoolExecutor) | Body rank ≥ 3 or batch calculations |
| `False`(default)  | Single-process serial | Small-scale or debugging |

**Note**: In parallel mode, you won't see detailed real-time progress bars in Jupyter, but the total elapsed time is displayed.

In [8]:
import time

# Serial
t0 = time.time()
_ = qcombo.easyCombo(2, 2, contraction=0, parallel=False,
                     show_process=True, savefile=False)
print(f"Serial (parallel=False): {time.time() - t0:.2f}s")

# Parallel
t0 = time.time()
_ = qcombo.easyCombo(2, 2, contraction=0, parallel=True,
                     show_process=True, savefile=False)
print(f"Parallel (parallel=True):  {time.time() - t0:.2f}s")

# Note: For low-body-rank commutators, parallel may be slower than serial due to process creation overhead.
# Generally, for [2,3] and above, parallel computing outperforms serial.

generalize wick caculating: [██████████████████████████████████████████████████] [63/63]100.0%  | Remaining: 0.0s
generalize wick caculating completed! Total Time:0.3s
generalize wick caculating: [██████████████████████████████████████████████████] [63/63]100.0%  | Remaining: 0.0s
generalize wick caculating completed! Total Time:0.3s
xiRule applying
xiRule completed!
natRule applying
natRule completed!
Contracting to 0-body terms
filtering: [██████████████████████████████████████████████████] [264/264]100.0%  | Remaining: 0.0s
filtering completed! Total Time:0.1s
canonicalizing: [██████████████████████████████████████████████████] [76/76]100.0%  | Remaining: 0.0s
canonicalizing completed! Total Time:0.4s
repetitive index simplifying: [██████████████████████████████████████████████████] [76/76]100.0%  | Remaining: 0.0s
repetitive index simplifying completed! Total Time:0.1s
Antisymmetry Simplify: [██████████████████████████████████████████████████] [76/76]100.0%  | Remaining: 0.0s
Antis

---
## 7. `show_process` Parameter: Control Output Verbosity

| Value | Behavior |
|-------|----------|
| `True` (default) | Show detailed progress bars, intermediate steps, file save info |
| `False` | Silent mode, only output key results |

In [9]:
print("show_process=True (default, show progress):")
_ = qcombo.easyCombo(1, 1, contraction=0,
                     show_process=True, parallel=False, savefile=False)

print("\n" + "=" * 50)
print("show_process=False (silent):")
_ = qcombo.easyCombo(1, 1, contraction=0,
                     show_process=False, parallel=False, savefile=False)
print("(no intermediate output)")

show_process=True (default, show progress):
generalize wick caculating: [██████████████████████████████████████████████████] [2/2]100.0%  | Remaining: 0.0s
generalize wick caculating completed! Total Time:0.0s
generalize wick caculating: [██████████████████████████████████████████████████] [2/2]100.0%  | Remaining: 0.0s
generalize wick caculating completed! Total Time:0.0s
xiRule applying
xiRule completed!
natRule applying
natRule completed!
Contracting to 0-body terms
filtering: [██████████████████████████████████████████████████] [4/4]100.0%  | Remaining: 0.0s
filtering completed! Total Time:0.0s
canonicalizing: [██████████████████████████████████████████████████] [2/2]100.0%  | Remaining: 0.0s
canonicalizing completed! Total Time:0.0s
repetitive index simplifying: [██████████████████████████████████████████████████] [2/2]100.0%  | Remaining: 0.0s
repetitive index simplifying completed! Total Time:0.0s
Antisymmetry Simplify: [██████████████████████████████████████████████████] [2/2]1

---
## 8. `savefile` / `latexOutput` / `amcOutput`: Output Files

`easyCombo` automatically generates two types of output files:

| File type | Purpose |
|-----------|---------|
| `.tex` | LaTeX-formatted analytic expression, can be directly compiled and viewed |
| `.amc` | AMC program input file, used for numerical computation |

**File naming convention**:
- MR: `commutator_{M}B{N}B_to_{k1}_{k2}_...B.tex`
- SR: `SR_commutator_{M}B{N}B_to_{k1}_{k2}_...B.tex`

In [10]:
# Do not save files (in-memory computation only)
print("savefile=False: no files generated")
r = qcombo.easyCombo(1, 1, contraction=0, savefile=False,
                     parallel=False, show_process=False)

# Save to custom filenames
print("\nsavefile=True + custom filenames:")
r = qcombo.easyCombo(1, 1, contraction=0, savefile=True,
                     latexOutput='my_commutator.tex',
                     amcOutput='my_commutator.amc',
                     parallel=False, show_process=False)
print("Files have been generated in the current directory.")

savefile=False: no files generated

savefile=True + custom filenames:
LaTeX output saved to: my_commutator.tex
AMC input file saved to: my_commutator.amc
Files have been generated in the current directory.


---
## 9. Return Value: `result.expr_dict` Explained

`expr_dict` is a dictionary with **keys** in the format `'{K}B_lambda{L}B'`:
- `K`: contraction body rank (0, 1, 2, ...)
- `L`: λ density matrix body rank (1, 2, ...)

Examples:
- `'0B_lambda1B'` → 0-body contraction, with 1-body λ
- `'2B_lambda2B'` → 2-body contraction, with 2-body λ

In [11]:
# View all contraction results for [2B, 2B]
r = qcombo.easyCombo(2, 2, contraction=[0, 1],
                     parallel=False, show_process=False, savefile=False)

print("Full content of expr_dict:")
for key, expr in r.expr_dict.items():
    if expr != 0:
        print(f"\n  [{key}]")
        jupyterDisplay(expr)

Full content of expr_dict:

  [0B_lambda1B]


<IPython.core.display.Latex object>


  [0B_lambda2B]


<IPython.core.display.Latex object>


  [0B_lambda3B]


<IPython.core.display.Latex object>


  [1B_lambda1B]


<IPython.core.display.Latex object>


  [1B_lambda2B]


<IPython.core.display.Latex object>

---
## 10. Complete Examples

### Example 1: Compute all contractions of $[2\text{B}, 2\text{B}]$ (MR mode)

This is the most common scenario: compute the commutator of two 2-body operators, obtaining analytic expressions for 0B, 1B, 2B, and 3B.

In [14]:
import time

print("Computing [2B, 2B] commutator (MR mode) ...")
t0 = time.time()
result = qcombo.easyCombo(2, 2,
                          wick_mode='MR',
                          parallel=False,
                          show_process=True,
                          savefile=True)
t1 = time.time()
print(f"\nTotal time: {t1 - t0:.2f}s")

print("\nNon-zero terms by contraction body rank:")
for key, expr in result.expr_dict.items():
    if expr != 0:
        jupyterDisplay(expr, f"{key}:")

Computing [2B, 2B] commutator (MR mode) ...
generalize wick caculating: [██████████████████████████████████████████████████] [63/63]100.0%  | Remaining: 0.0s
generalize wick caculating completed! Total Time:0.6s
generalize wick caculating: [██████████████████████████████████████████████████] [63/63]100.0%  | Remaining: 0.0s
generalize wick caculating completed! Total Time:0.4s
xiRule applying
xiRule completed!
natRule applying
natRule completed!
Contracting to 0-body terms
filtering: [██████████████████████████████████████████████████] [264/264]100.0%  | Remaining: 0.0s
filtering completed! Total Time:0.2s
canonicalizing: [██████████████████████████████████████████████████] [76/76]100.0%  | Remaining: 0.0s
canonicalizing completed! Total Time:0.6s
repetitive index simplifying: [██████████████████████████████████████████████████] [76/76]100.0%  | Remaining: 0.0s
repetitive index simplifying completed! Total Time:0.1s
Antisymmetry Simplify: [██████████████████████████████████████████████

<IPython.core.display.Latex object>

0B_lambda2B:


<IPython.core.display.Latex object>

0B_lambda3B:


...

<IPython.core.display.Latex object>

2B_lambda1B:


<IPython.core.display.Latex object>

3B_lambda1B:


<IPython.core.display.Latex object>

### Example 2: Compute $[1\text{B}, 2\text{B}]$ in SR mode

In [13]:
print("Computing [1B, 2B] commutator (SR mode) ...")
result = qcombo.easyCombo(1, 2,
                          wick_mode='SR',
                          parallel=False,
                          show_process=False,
                          savefile=False)

print("\nNon-zero results in SR mode:")
for key, expr in result.expr_dict.items():
    if expr != 0:
        jupyterDisplay(expr, f"{key}:")

Computing [1B, 2B] commutator (SR mode) ...

Non-zero results in SR mode:
1B_lambda1B:


<IPython.core.display.Latex object>

2B_lambda1B:


<IPython.core.display.Latex object>

---
## 13. Summary

### Common Usage Patterns

```python
import qcombo

# Simplest: auto everything
r = qcombo.easyCombo(2, 2)

# Only output 0-body, don't save files
r = qcombo.easyCombo(2, 2, contraction=0, savefile=False)

# SR mode
r = qcombo.easyCombo(1, 2, wick_mode='SR')

# Silent + serial (for debugging)
r = qcombo.easyCombo(2, 2, show_process=False, parallel=False, savefile=False)

# Access results
for key, expr in r.expr_dict.items():
    print(key, "→", expr)
```

### Parameter Overview

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `left` | int or list | **(required)** | Left operator body rank / indices |
| `right` | int or list | **(required)** | Right operator body rank / indices |
| `contraction` | int / list / None | `None` | Contraction body rank |
| `wick_mode` | `'MR'` / `'SR'` | `'MR'` | Wick mode |
| `show_process` | bool | `True` | Show progress |
| `parallel` | bool | `False` | Parallel computation |
| `savefile` | bool | `True` | Save output files |
| `latexOutput` | str / None | `None` | `.tex` filename |
| `amcOutput` | str / None | `None` | `.amc` filename |

### Related Utility Functions

| Function | Purpose |
|----------|---------|
| `qcombo.texExp(expr)` | SymPy expression → LaTeX string |
| `qcombo.simplifyUseBoth(expr)` | Antisymmetry + dummy index simplification |
| `qcombo.MergeSameMatrixElement(expr)` | Merge identical matrix elements |
| `qcombo.filterLambdaBody(expr, n)` | Filter by λ body rank |
| `getEquationLatexStrFromExpr(expr, ...)` | Generate LaTeX equation |
| `getEquationAmcStrFromExpr(expr, ...)` | Generate AMC input |